<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Система управления задачами проекта</h2>

----

### Вариант задания 23


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Проект представляет собой систему управления задачами.

Базовый класс Task содержит общие данные задачи:
- идентификатор задачи (TaskId);
- название (TaskName);
- приоритет (Priority);
- исполнитель (AssignedTo).

От класса Task наследуются:
- DelegateTask — делегированная задача;
- TeamTask — командная задача;
- ResearchTask — исследовательская задача.

Каждый производный класс добавляет собственные свойства и методы и переопределяет отдельные методы базового класса.

#### Дополнительное задание
Добавьте к существующим классам (базовому и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;

public interface ITask
{
    int TaskId { get; set; }
    string TaskName { get; set; }
    string Priority { get; set; }
    string AssignedTo { get; set; }

    void Execute();
    void GetTaskDetails();
}

public interface INotifiable
{
    void Notify(string recipient, string message);
}

public interface ITaskNotifier
{
    void SendNotification(string message);
}

public class ConsoleTaskNotifier : ITaskNotifier
{
    public void SendNotification(string message)
    {
        Console.WriteLine($"[СИСТЕМА УВЕДОМЛЕНИЙ] {message}");
    }
}

public class Task : ITask, INotifiable
{
    public int TaskId { get; set; }
    public string TaskName { get; set; }
    public string Priority { get; set; }
    public string AssignedTo { get; set; }

    public DateTime CreatedAt { get; set; }
    public bool IsCompleted { get; set; }
    public int EstimatedHours { get; set; }

    public Task(int taskId, string taskName, string priority, int estimatedHours = 8)
    {
        TaskId = taskId;
        TaskName = taskName;
        Priority = priority;
        AssignedTo = "Не назначен";
        CreatedAt = DateTime.Now;
        IsCompleted = false;
        EstimatedHours = estimatedHours;
    }

    public virtual void Execute()
    {
        Console.WriteLine($"Выполнение базовой задачи #{TaskId} "{TaskName}"");
    }

    void INotifiable.Notify(string recipient, string message)
    {
        Console.WriteLine($"[Явный INotifiable] Уведомление для {recipient} по задаче #{TaskId}: {message}");
    }

    public virtual void MarkAsComplete()
    {
        IsCompleted = true;
        Console.WriteLine($"Задача "{TaskName}" завершена.");
    }

    public virtual void GetTaskDetails()
    {
        Console.WriteLine($"ID: {TaskId}");
        Console.WriteLine($"Название: {TaskName}");
        Console.WriteLine($"Приоритет: {Priority}");
        Console.WriteLine($"Исполнитель: {AssignedTo}");
        Console.WriteLine($"Оценка времени: {EstimatedHours} ч.");
        Console.WriteLine($"Статус: {(IsCompleted ? "Завершена" : "В работе")}");
    }

    public virtual void ReassignTo(string person)
    {
        AssignedTo = person;
        Console.WriteLine($"Задача "{TaskName}" назначена: {person}");
    }

    public void ReassignTo(string person, string reason)
    {
        AssignedTo = person;
        Console.WriteLine($"Задача "{TaskName}" назначена: {person}. Причина: {reason}");
    }
}

public class DelegateTask : Task, ITask
{
    public DateTime DueDate { get; set; }
    public string DelegatedBy { get; set; }
    public string ApprovalStatus { get; set; }

    public DelegateTask(int taskId, string taskName, string priority, DateTime dueDate, string delegatedBy, int estimatedHours = 12)
        : base(taskId, taskName, priority, estimatedHours)
    {
        DueDate = dueDate;
        DelegatedBy = delegatedBy;
        ApprovalStatus = "Ожидает согласования";
    }

    void ITask.Execute()
    {
        Console.WriteLine($"[ITask.Execute (Явная реализация)] Делегированная задача "{TaskName}" выполняется с согласованием от {DelegatedBy}.");
    }

    public override void MarkAsComplete()
    {
        base.MarkAsComplete();
        ApprovalStatus = "Утверждено";
        Console.WriteLine($"Срок сдачи: {DueDate:d}. Статус утверждения: {ApprovalStatus}");
    }

    public void ShowDeadline()
    {
        Console.WriteLine($"Дедлайн делегированной задачи: {DueDate:d}");
    }

    public void ExtendDeadline(int days)
    {
        DueDate = DueDate.AddDays(days);
        Console.WriteLine($"Срок задачи "{TaskName}" продлен на {days} дн. Новый дедлайн: {DueDate:d}");
    }
}

public class TeamTask : Task, ITask
{
    public string TeamName { get; set; }
    public int MemberCount { get; set; }
    public int SprintNumber { get; set; }

    public TeamTask(int taskId, string taskName, string priority, string teamName, int memberCount, int sprintNumber = 4)
        : base(taskId, taskName, priority, 40)
    {
        TeamName = teamName;
        MemberCount = memberCount;
        SprintNumber = sprintNumber;
    }

    void ITask.Execute()
    {
        Console.WriteLine($"[ITask.Execute (Явная реализация)] Команда "{TeamName}" ({MemberCount} чел.) выполняет спринт #{SprintNumber}.");
    }

    public override void ReassignTo(string person)
    {
        AssignedTo = person;
        Console.WriteLine($"Задача "{TaskName}" команды "{TeamName}" назначена тимлиду: {person}");
    }

    public void AddMember()
    {
        MemberCount++;
        Console.WriteLine($"Команда "{TeamName}": участник добавлен. Всего: {MemberCount}");
    }

    public void ChangeSprint(int newSprint)
    {
        SprintNumber = newSprint;
        Console.WriteLine($"Задача перенесена в спринт #{SprintNumber}");
    }
}

public class ResearchTask : Task, ITask
{
    public string DataSource { get; set; }
    public string ResearchMethod { get; set; }
    public string Findings { get; set; }

    public ResearchTask(int taskId, string taskName, string priority, string dataSource, string researchMethod)
        : base(taskId, taskName, priority, 20)
    {
        DataSource = dataSource;
        ResearchMethod = researchMethod;
        Findings = "Исследование не завершено";
    }

    void ITask.Execute()
    {
        Console.WriteLine($"[ITask.Execute (Явная реализация)] Исследовательская задача "{TaskName}" запущена по методу: {ResearchMethod}.");
    }

    public override void GetTaskDetails()
    {
        base.GetTaskDetails();
        Console.WriteLine($"Источник данных: {DataSource}");
        Console.WriteLine($"Метод исследования: {ResearchMethod}");
        Console.WriteLine($"Результаты: {Findings}");
    }

    public void AddFindings(string findings)
    {
        Findings = findings;
        Console.WriteLine($"Результаты исследования зафиксированы: {Findings}");
    }

    public void ChangeResearchMethod(string method)
    {
        ResearchMethod = method;
        Console.WriteLine($"Новый метод исследования: {ResearchMethod}");
    }
}

public class TaskManagerService
{
    private readonly ITaskNotifier _notifier;

    public TaskManagerService(ITaskNotifier notifier)
    {
        _notifier = notifier ?? throw new ArgumentNullException(nameof(notifier));
    }

    public void ProcessTask(ITask task)
    {
        _notifier.SendNotification($"Начата обработка задачи #{task.TaskId}: "{task.TaskName}" (Приоритет: {task.Priority})");
        task.Execute();
        _notifier.SendNotification($"Задача #{task.TaskId} успешно обработана сервисом.
");
    }
}

Console.WriteLine("================ СИСТЕМА УПРАВЛЕНИЯ ЗАДАЧАМИ (ВАРИАНТ 23) ================\n");

DelegateTask task1 = new DelegateTask(1, "Подготовить отчет", "Высокий", DateTime.Today.AddDays(5), "Руководитель проекта");
TeamTask task2 = new TeamTask(2, "Разработать модуль авторизации", "Критический", "Backend Devs", 5, 4);
ResearchTask task3 = new ResearchTask(3, "Анализ архитектуры микросервисов", "Средний", "IEEE / ACM Digital Library", "Сравнительный бенчмарк");

Console.WriteLine("--- 1. Явная реализация интерфейсов ---");
INotifiable notifiable = task1;
notifiable.Notify("Иван Иванов", "Пожалуйста, согласуйте сроки выполнения.");

ITask itask1 = task1;
itask1.Execute();

ITask itask2 = task2;
itask2.Execute();

ITask itask3 = task3;
itask3.Execute();

Console.WriteLine();

Console.WriteLine("--- 2. Управление зависимостями (Dependency Injection) ---");
ITaskNotifier notifier = new ConsoleTaskNotifier();
TaskManagerService taskManager = new TaskManagerService(notifier);

taskManager.ProcessTask(task1);
taskManager.ProcessTask(task2);
taskManager.ProcessTask(task3);

Console.WriteLine("--- 3. Специализированные методы классов ---");
task1.ExtendDeadline(3);
task1.MarkAsComplete();

Console.WriteLine();
task2.AddMember();
task2.ReassignTo("Алексей");

Console.WriteLine();
task3.AddFindings("Выбрана брокерная архитектура на базе RabbitMQ.");
task3.GetTaskDetails();
